# Notebook 19. Cleaned-Cluster Date Pairing and Sequencing

This notebook follows the cleaned `k = 2` subtype framework and asks the next professor-guided question:

**Do the cleaned clusters tend to occur near each other in time, suggesting a before/after synoptic sequence, or do they behave more like distinct event families?**

Primary goals:

- use the cleaned `k = 2` labels from `Notebook 15`
- quantify cross-cluster event pairing within `±1`, `±2`, and `±3` days
- summarize lead/lag direction between cleaned `Cluster 1` and cleaned `Cluster 2`
- save representative close pairs for manual follow-up

Secondary overlay:

- if `Notebook 18` quartile selection is available, attach `lower / middle / upper` tags to cleaned `Cluster 1` events
- test whether cleaned `Cluster 2` events tend to occur near the stronger or weaker ends of cleaned `Cluster 1`

This notebook is intentionally lightweight: it uses the cleaned event catalog and quartile-selection tables only, with no ERA5 reprocessing.


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from jpcz_catalog.analysis import add_japan_local_time_columns

CLEANED_RUN_EXPORT_DIR = Path("outputs/verification/objective_subtype_low_level_cleaned_sensitivity")
QUARTILE_EXPORT_DIR = Path("outputs/verification/objective_subtype_cleaned_cluster_quartiles")
PAIRING_EXPORT_DIR = Path("outputs/verification/objective_subtype_cleaned_cluster_date_pairing")
PAIRING_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_cleaned_cluster_date_pairing_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CLUSTERED_EVENTS_PATH = CLEANED_RUN_EXPORT_DIR / "clustered_events_cleaned_low_level_k2_k3_k4.csv"
NOTEBOOK15_SOLUTION_SUMMARY_PATH = CLEANED_RUN_EXPORT_DIR / "cleaned_low_level_solution_summary.csv"
QUARTILE_SELECTION_PATH = QUARTILE_EXPORT_DIR / "cleaned_k2_cluster1_quartile_selection.csv"
QUARTILE_SUMMARY_PATH = QUARTILE_EXPORT_DIR / "cleaned_k2_cluster1_quartile_summary.csv"

PRIMARY_CLUSTER_COLUMN = "cleaned_cluster_ward_2"
EXPLORATORY_CLUSTER_COLUMN = "cleaned_cluster_ward_3"
PRIMARY_CLUSTER_A_ID = 1
PRIMARY_CLUSTER_B_ID = 2
PAIR_WINDOWS_DAYS = [1, 2, 3]
MAX_PAIR_WINDOW_DAYS = max(PAIR_WINDOWS_DAYS)
MAX_REPRESENTATIVE_PAIRS = 30
SAVE_PLOTS = True
NDJF_MONTH_ORDER = [11, 12, 1, 2]

ANNOTATED_EVENTS_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_events_with_quartile_tags.csv"
PAIR_SUMMARY_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_cross_cluster_pair_summary_by_window.csv"
QUARTILE_PAIR_SUMMARY_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_cluster1_quartile_pair_summary_by_window.csv"
ALL_PAIRS_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_all_cross_cluster_pairs.csv"
PAIRS_WITHIN_MAX_WINDOW_PATH = PAIRING_EXPORT_DIR / f"cleaned_k2_cross_cluster_pairs_within_{MAX_PAIR_WINDOW_DAYS}d.csv"
NEAREST_CLUSTER1_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_cluster1_nearest_cluster2_pairs.csv"
NEAREST_CLUSTER2_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_cluster2_nearest_cluster1_pairs.csv"
REPRESENTATIVE_PAIRS_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_representative_close_pairs.csv"
MONTHLY_CLUSTER_COUNTS_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_monthly_cluster_counts.csv"
MONTHLY_CLUSTER1_QUARTILE_COUNTS_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_cluster1_monthly_quartile_counts.csv"
PLOT_INVENTORY_PATH = PAIRING_EXPORT_DIR / "cleaned_k2_date_pairing_plot_inventory.csv"


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None



def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True



def ordinal_word(value: int) -> str:
    lookup = {1: "first", 2: "second", 3: "third", 4: "fourth", 5: "fifth"}
    return lookup.get(value, f"{value}th")



def size_rank_descriptor(rank: int, total: int) -> str:
    if total <= 1:
        return "only subgroup"
    if rank == 1:
        return "largest subgroup"
    if rank == total:
        return "smallest subgroup"
    return f"{ordinal_word(rank)}-largest subgroup"



def build_cluster_labels_from_counts(cluster_counts: pd.Series | dict[int, int]):
    counts_dict = {int(cluster_id): int(n_events) for cluster_id, n_events in dict(cluster_counts).items()}
    ranked = sorted(counts_dict.items(), key=lambda item: (-item[1], item[0]))
    rank_lookup = {cluster_id: rank for rank, (cluster_id, _) in enumerate(ranked, start=1)}
    total = len(ranked)
    long_labels = {}
    rows = []
    for cluster_id, n_events in sorted(counts_dict.items()):
        descriptor = size_rank_descriptor(rank_lookup[cluster_id], total)
        long_labels[cluster_id] = f"Cluster {cluster_id}: n={n_events} ({descriptor})"
        rows.append(
            {
                "cluster_id": cluster_id,
                "n_events": n_events,
                "size_rank": rank_lookup[cluster_id],
                "size_descriptor": descriptor,
                "cluster_label": long_labels[cluster_id],
            }
        )
    return long_labels, pd.DataFrame(rows)



def month_label(month_number: int) -> str:
    lookup = {11: "Nov", 12: "Dec", 1: "Jan", 2: "Feb"}
    return lookup.get(int(month_number), str(month_number))



def build_all_cross_cluster_pairs(cluster1_df: pd.DataFrame, cluster2_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row1 in cluster1_df.itertuples(index=False):
        for row2 in cluster2_df.itertuples(index=False):
            lag_hours = (row2.event_peak - row1.event_peak).total_seconds() / 3600.0
            rows.append(
                {
                    "cluster1_event_index": int(row1.event_index),
                    "cluster1_event_peak": row1.event_peak,
                    "cluster1_event_peak_jst": row1.event_peak_jst,
                    "cluster1_duration_hours": float(row1.duration_hours) if pd.notna(row1.duration_hours) else np.nan,
                    "cluster1_quartile_subset": row1.cluster1_quartile_subset,
                    "cluster2_event_index": int(row2.event_index),
                    "cluster2_event_peak": row2.event_peak,
                    "cluster2_event_peak_jst": row2.event_peak_jst,
                    "cluster2_duration_hours": float(row2.duration_hours) if pd.notna(row2.duration_hours) else np.nan,
                    "lag_hours_cluster2_minus_cluster1": lag_hours,
                    "abs_lag_hours": abs(lag_hours),
                    "cluster2_after_cluster1": lag_hours > 0.0,
                }
            )
    pair_df = pd.DataFrame(rows)
    if pair_df.empty:
        return pair_df
    pair_df["cluster1_event_peak"] = pd.to_datetime(pair_df["cluster1_event_peak"])
    pair_df["cluster2_event_peak"] = pd.to_datetime(pair_df["cluster2_event_peak"])
    pair_df["cluster1_event_peak_jst"] = pd.to_datetime(pair_df["cluster1_event_peak_jst"])
    pair_df["cluster2_event_peak_jst"] = pd.to_datetime(pair_df["cluster2_event_peak_jst"])
    return pair_df.sort_values(["abs_lag_hours", "cluster1_event_peak", "cluster2_event_peak"]).reset_index(drop=True)



def summarize_pairs_by_window(pair_df: pd.DataFrame, cluster1_total: int, cluster2_total: int, windows_days: list[int]) -> pd.DataFrame:
    rows = []
    for window_days in windows_days:
        max_hours = float(window_days * 24)
        subset = pair_df.loc[pair_df["abs_lag_hours"] <= max_hours].copy()
        cluster1_paired = subset["cluster1_event_index"].nunique()
        cluster2_paired = subset["cluster2_event_index"].nunique()
        rows.append(
            {
                "window_days": window_days,
                "max_abs_lag_hours": max_hours,
                "n_cross_cluster_pairs": len(subset),
                "n_cluster1_events_with_partner": cluster1_paired,
                "n_cluster2_events_with_partner": cluster2_paired,
                "percent_cluster1_events_with_partner": 100.0 * cluster1_paired / cluster1_total if cluster1_total else np.nan,
                "percent_cluster2_events_with_partner": 100.0 * cluster2_paired / cluster2_total if cluster2_total else np.nan,
                "n_pairs_cluster2_after_cluster1": int((subset["lag_hours_cluster2_minus_cluster1"] > 0.0).sum()),
                "n_pairs_cluster2_before_cluster1": int((subset["lag_hours_cluster2_minus_cluster1"] < 0.0).sum()),
                "median_abs_lag_hours": float(subset["abs_lag_hours"].median()) if not subset.empty else np.nan,
                "median_signed_lag_hours_cluster2_minus_cluster1": float(subset["lag_hours_cluster2_minus_cluster1"].median()) if not subset.empty else np.nan,
            }
        )
    return pd.DataFrame(rows)



def summarize_quartile_pairing_by_window(pair_df: pd.DataFrame, cluster1_df: pd.DataFrame, windows_days: list[int]) -> pd.DataFrame:
    if pair_df.empty or "cluster1_quartile_subset" not in pair_df.columns:
        return pd.DataFrame()
    subset_order = ["lower", "middle", "upper"]
    rows = []
    for window_days in windows_days:
        max_hours = float(window_days * 24)
        subset = pair_df.loc[pair_df["abs_lag_hours"] <= max_hours].copy()
        for quartile_subset in subset_order:
            source_df = cluster1_df.loc[cluster1_df["cluster1_quartile_subset"] == quartile_subset].copy()
            source_total = len(source_df)
            paired_subset = subset.loc[subset["cluster1_quartile_subset"] == quartile_subset].copy()
            paired_events = paired_subset["cluster1_event_index"].nunique()
            rows.append(
                {
                    "window_days": window_days,
                    "cluster1_quartile_subset": quartile_subset,
                    "n_cluster1_events_in_subset": source_total,
                    "n_cluster1_events_with_cluster2_partner": paired_events,
                    "percent_cluster1_events_with_partner": 100.0 * paired_events / source_total if source_total else np.nan,
                    "n_cross_cluster_pairs": len(paired_subset),
                    "median_abs_lag_hours": float(paired_subset["abs_lag_hours"].median()) if not paired_subset.empty else np.nan,
                    "n_pairs_cluster2_after_cluster1": int((paired_subset["lag_hours_cluster2_minus_cluster1"] > 0.0).sum()),
                    "n_pairs_cluster2_before_cluster1": int((paired_subset["lag_hours_cluster2_minus_cluster1"] < 0.0).sum()),
                }
            )
    return pd.DataFrame(rows)



def build_nearest_pair_tables(pair_df: pd.DataFrame, *, max_window_days: int):
    if pair_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    nearest_cluster1_df = (
        pair_df.sort_values(["cluster1_event_index", "abs_lag_hours", "cluster2_event_peak"])
        .drop_duplicates(subset=["cluster1_event_index"], keep="first")
        .reset_index(drop=True)
    )
    nearest_cluster2_df = (
        pair_df.sort_values(["cluster2_event_index", "abs_lag_hours", "cluster1_event_peak"])
        .drop_duplicates(subset=["cluster2_event_index"], keep="first")
        .reset_index(drop=True)
    )
    representative_pairs_df = pd.concat(
        [
            nearest_cluster1_df.assign(selection_source="nearest_to_cluster1"),
            nearest_cluster2_df.assign(selection_source="nearest_to_cluster2"),
        ],
        ignore_index=True,
    )
    representative_pairs_df["pair_key"] = representative_pairs_df.apply(
        lambda row: f"{min(int(row['cluster1_event_index']), int(row['cluster2_event_index']))}_{max(int(row['cluster1_event_index']), int(row['cluster2_event_index']))}",
        axis=1,
    )
    representative_pairs_df = representative_pairs_df.sort_values(["abs_lag_hours", "selection_source"]).drop_duplicates(subset=["pair_key"], keep="first")
    representative_pairs_df["within_max_window"] = representative_pairs_df["abs_lag_hours"] <= float(max_window_days * 24)
    representative_pairs_df = representative_pairs_df.sort_values(["within_max_window", "abs_lag_hours"], ascending=[False, True]).reset_index(drop=True)
    return nearest_cluster1_df, nearest_cluster2_df, representative_pairs_df.head(MAX_REPRESENTATIVE_PAIRS).copy()



def plot_monthly_cluster_counts(monthly_counts_df: pd.DataFrame):
    cluster_ids = sorted(monthly_counts_df["cluster_id"].unique().tolist())
    available_months = set(monthly_counts_df["event_month"].astype(int).tolist())
    month_order = [month for month in NDJF_MONTH_ORDER if month in available_months]
    month_labels = [month_label(month) for month in month_order]
    x = np.arange(len(month_order))
    width = 0.35 if len(cluster_ids) == 2 else 0.8 / max(len(cluster_ids), 1)
    fig, ax = plt.subplots(figsize=(9.2, 4.8))
    for idx, cluster_id in enumerate(cluster_ids):
        subset = monthly_counts_df.loc[monthly_counts_df["cluster_id"] == cluster_id].sort_values("event_month")
        offset = (idx - (len(cluster_ids) - 1) / 2.0) * width
        ax.bar(x + offset, subset["n_events"], width=width, label=f"Cluster {cluster_id}")
    ax.set_xticks(x)
    ax.set_xticklabels(month_labels)
    ax.set_xlabel("Month")
    ax.set_ylabel("Event count")
    ax.set_title("Cleaned k=2 monthly event counts")
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig



def plot_quartile_pairing_rates(quartile_pairing_summary_df: pd.DataFrame):
    subset_order = ["lower", "middle", "upper"]
    window_order = quartile_pairing_summary_df["window_days"].drop_duplicates().tolist()
    x = np.arange(len(window_order))
    width = 0.24
    fig, ax = plt.subplots(figsize=(9.4, 4.8))
    colors = {"lower": "#1f78b4", "middle": "#9e9e9e", "upper": "#d95f02"}
    for idx, subset_name in enumerate(subset_order):
        subset = quartile_pairing_summary_df.loc[quartile_pairing_summary_df["cluster1_quartile_subset"] == subset_name].sort_values("window_days")
        if subset.empty:
            continue
        offset = (idx - 1) * width
        ax.bar(x + offset, subset["percent_cluster1_events_with_partner"], width=width, label=subset_name.capitalize(), color=colors.get(subset_name))
    ax.set_xticks(x)
    ax.set_xticklabels([f"±{window} d" for window in window_order])
    ax.set_xlabel("Pairing window")
    ax.set_ylabel("Cluster 1 subset events paired with Cluster 2 [%]")
    ax.set_title("Cluster 2 pairing rate by Cluster 1 quartile subset")
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig



def plot_nearest_lag_histogram(nearest_cluster1_df: pd.DataFrame, *, max_window_days: int):
    max_hours = float(max_window_days * 24)
    subset = nearest_cluster1_df.loc[nearest_cluster1_df["abs_lag_hours"] <= max_hours].copy()
    fig, ax = plt.subplots(figsize=(9.4, 4.8))
    if subset.empty:
        ax.text(0.5, 0.5, f"No Cluster 1 to Cluster 2 pairings within ±{max_window_days} days", ha="center", va="center", transform=ax.transAxes)
    else:
        bins = np.arange(-max_hours - 6, max_hours + 12, 12)
        ax.hist(subset["lag_hours_cluster2_minus_cluster1"], bins=bins, color="#6a3d9a", edgecolor="white")
        ax.axvline(0.0, color="black", linewidth=1.0, linestyle="--")
    ax.set_xlabel("Lag hours (Cluster 2 minus Cluster 1)")
    ax.set_ylabel("Cluster 1 events")
    ax.set_title(f"Nearest Cluster 2 partner for each Cluster 1 event within ±{max_window_days} days")
    fig.tight_layout()
    return fig


In [ ]:
paths_to_restore = [
    CLEANED_CLUSTERED_EVENTS_PATH,
    NOTEBOOK15_SOLUTION_SUMMARY_PATH,
    QUARTILE_SELECTION_PATH,
    QUARTILE_SUMMARY_PATH,
]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

if not CLEANED_CLUSTERED_EVENTS_PATH.exists():
    raise RuntimeError("Run Notebook 15 first so the cleaned clustered event table exists.")

clustered_df = pd.read_csv(CLEANED_CLUSTERED_EVENTS_PATH)
clustered_df["event_index"] = clustered_df.index.astype(int)
clustered_df["event_peak"] = pd.to_datetime(clustered_df["event_peak"])
clustered_df = add_japan_local_time_columns(clustered_df)
required_columns = [PRIMARY_CLUSTER_COLUMN, EXPLORATORY_CLUSTER_COLUMN, "event_peak", "event_peak_jst"]
missing_columns = [column for column in required_columns if column not in clustered_df.columns]
if missing_columns:
    raise RuntimeError(f"Missing required columns in the cleaned clustered event table: {missing_columns}")

cluster_counts = clustered_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).value_counts().sort_index()
cluster_label_lookup, cluster_label_df = build_cluster_labels_from_counts(cluster_counts)
missing_target_clusters = [cluster_id for cluster_id in [PRIMARY_CLUSTER_A_ID, PRIMARY_CLUSTER_B_ID] if cluster_id not in cluster_label_lookup]
if missing_target_clusters:
    raise RuntimeError(f"Missing cleaned k=2 cluster ids needed for Notebook 19: {missing_target_clusters}")

cluster1_label = cluster_label_lookup[PRIMARY_CLUSTER_A_ID]
cluster2_label = cluster_label_lookup[PRIMARY_CLUSTER_B_ID]
clustered_df["cluster_label_k2"] = clustered_df[PRIMARY_CLUSTER_COLUMN].astype(int).map(cluster_label_lookup)
clustered_df["cluster1_quartile_subset"] = np.where(
    clustered_df[PRIMARY_CLUSTER_COLUMN].astype(int) == PRIMARY_CLUSTER_A_ID,
    "middle",
    "not_cluster1",
)
quartile_overlay_available = False
quartile_selection_df = None
quartile_summary_df = None

if QUARTILE_SELECTION_PATH.exists():
    quartile_selection_df = pd.read_csv(QUARTILE_SELECTION_PATH)
    if "event_index" not in quartile_selection_df.columns or "quartile_subset" not in quartile_selection_df.columns:
        raise RuntimeError("Notebook 18 quartile selection table is missing required columns.")
    quartile_selection_df["event_index"] = quartile_selection_df["event_index"].astype(int)
    quartile_lookup = quartile_selection_df.set_index("event_index")["quartile_subset"].to_dict()
    clustered_df.loc[clustered_df[PRIMARY_CLUSTER_COLUMN].astype(int) == PRIMARY_CLUSTER_A_ID, "cluster1_quartile_subset"] = clustered_df.loc[
        clustered_df[PRIMARY_CLUSTER_COLUMN].astype(int) == PRIMARY_CLUSTER_A_ID, "event_index"
    ].map(quartile_lookup).fillna("middle")
    quartile_overlay_available = True

if QUARTILE_SUMMARY_PATH.exists():
    quartile_summary_df = pd.read_csv(QUARTILE_SUMMARY_PATH)

cluster1_df = clustered_df.loc[clustered_df[PRIMARY_CLUSTER_COLUMN].astype(int) == PRIMARY_CLUSTER_A_ID].copy().sort_values("event_peak")
cluster2_df = clustered_df.loc[clustered_df[PRIMARY_CLUSTER_COLUMN].astype(int) == PRIMARY_CLUSTER_B_ID].copy().sort_values("event_peak")

quartile_count_df = (
    cluster1_df["cluster1_quartile_subset"].value_counts().rename_axis("cluster1_quartile_subset").reset_index(name="n_events")
)
quartile_count_df["cluster1_quartile_subset"] = pd.Categorical(quartile_count_df["cluster1_quartile_subset"], categories=["lower", "middle", "upper", "not_cluster1"], ordered=True)
quartile_count_df = quartile_count_df.sort_values("cluster1_quartile_subset").reset_index(drop=True)

annotated_clustered_df = clustered_df.copy()
annotated_clustered_df.to_csv(ANNOTATED_EVENTS_PATH, index=False)
maybe_copy_to_drive(ANNOTATED_EVENTS_PATH)

context_summary_df = pd.DataFrame(
    [
        {"metric": "primary_cluster_column", "value": PRIMARY_CLUSTER_COLUMN},
        {"metric": "cluster1_label", "value": cluster1_label},
        {"metric": "cluster2_label", "value": cluster2_label},
        {"metric": "cluster1_event_count", "value": len(cluster1_df)},
        {"metric": "cluster2_event_count", "value": len(cluster2_df)},
        {"metric": "pair_windows_days", "value": ", ".join(str(value) for value in PAIR_WINDOWS_DAYS)},
        {"metric": "quartile_overlay_available", "value": quartile_overlay_available},
    ]
)

print("Cleaned cluster-date pairing context")
display(cluster_label_df)
display(context_summary_df)
print("\nCluster 1 quartile-subset counts")
display(quartile_count_df)


In [ ]:
required_globals = [
    "clustered_df",
    "cluster1_df",
    "cluster2_df",
    "cluster1_label",
    "cluster2_label",
    "quartile_overlay_available",
    "quartile_count_df",
    "maybe_copy_to_drive",
    "build_all_cross_cluster_pairs",
    "summarize_pairs_by_window",
    "summarize_quartile_pairing_by_window",
    "build_nearest_pair_tables",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 19 setup, config/helper, and cleaned-cluster context cells before the pairing-analysis cell. "
        f"Missing globals: {missing_globals}"
    )

all_pairs_df = build_all_cross_cluster_pairs(cluster1_df, cluster2_df)
if all_pairs_df.empty:
    raise RuntimeError("No cross-cluster pairs could be constructed from the cleaned k=2 catalog.")

pair_window_summary_df = summarize_pairs_by_window(all_pairs_df, len(cluster1_df), len(cluster2_df), PAIR_WINDOWS_DAYS)
quartile_pairing_summary_df = summarize_quartile_pairing_by_window(all_pairs_df, cluster1_df, PAIR_WINDOWS_DAYS)
nearest_cluster1_df, nearest_cluster2_df, representative_pairs_df = build_nearest_pair_tables(all_pairs_df, max_window_days=MAX_PAIR_WINDOW_DAYS)

pairs_within_max_window_df = all_pairs_df.loc[all_pairs_df["abs_lag_hours"] <= float(MAX_PAIR_WINDOW_DAYS * 24)].copy().reset_index(drop=True)
all_pairs_df.to_csv(ALL_PAIRS_PATH, index=False)
maybe_copy_to_drive(ALL_PAIRS_PATH)
pairs_within_max_window_df.to_csv(PAIRS_WITHIN_MAX_WINDOW_PATH, index=False)
maybe_copy_to_drive(PAIRS_WITHIN_MAX_WINDOW_PATH)
pair_window_summary_df.to_csv(PAIR_SUMMARY_PATH, index=False)
maybe_copy_to_drive(PAIR_SUMMARY_PATH)
nearest_cluster1_df.to_csv(NEAREST_CLUSTER1_PATH, index=False)
maybe_copy_to_drive(NEAREST_CLUSTER1_PATH)
nearest_cluster2_df.to_csv(NEAREST_CLUSTER2_PATH, index=False)
maybe_copy_to_drive(NEAREST_CLUSTER2_PATH)
representative_pairs_df.to_csv(REPRESENTATIVE_PAIRS_PATH, index=False)
maybe_copy_to_drive(REPRESENTATIVE_PAIRS_PATH)

if quartile_overlay_available and not quartile_pairing_summary_df.empty:
    quartile_pairing_summary_df.to_csv(QUARTILE_PAIR_SUMMARY_PATH, index=False)
    maybe_copy_to_drive(QUARTILE_PAIR_SUMMARY_PATH)

monthly_cluster_counts_df = (
    clustered_df.assign(event_month=clustered_df["event_peak"].dt.month)
    .groupby([PRIMARY_CLUSTER_COLUMN, "cluster_label_k2", "event_month"], as_index=False)
    .size()
    .rename(columns={PRIMARY_CLUSTER_COLUMN: "cluster_id", "size": "n_events"})
    .assign(month_order=lambda df: df["event_month"].map({11:0, 12:1, 1:2, 2:3})).sort_values(["month_order", "cluster_id"]).drop(columns="month_order")
    .reset_index(drop=True)
)
monthly_cluster_counts_df.to_csv(MONTHLY_CLUSTER_COUNTS_PATH, index=False)
maybe_copy_to_drive(MONTHLY_CLUSTER_COUNTS_PATH)

monthly_cluster1_quartile_counts_df = (
    cluster1_df.assign(event_month=cluster1_df["event_peak"].dt.month)
    .groupby(["cluster1_quartile_subset", "event_month"], as_index=False)
    .size()
    .rename(columns={"size": "n_events"})
    .assign(month_order=lambda df: df["event_month"].map({11:0, 12:1, 1:2, 2:3})).sort_values(["month_order", "cluster1_quartile_subset"]).drop(columns="month_order")
    .reset_index(drop=True)
)
monthly_cluster1_quartile_counts_df.to_csv(MONTHLY_CLUSTER1_QUARTILE_COUNTS_PATH, index=False)
maybe_copy_to_drive(MONTHLY_CLUSTER1_QUARTILE_COUNTS_PATH)

pair_window_summary_df["percent_cluster1_events_with_partner"] = pair_window_summary_df["percent_cluster1_events_with_partner"].round(2)
pair_window_summary_df["percent_cluster2_events_with_partner"] = pair_window_summary_df["percent_cluster2_events_with_partner"].round(2)
pair_window_summary_df["median_abs_lag_hours"] = pair_window_summary_df["median_abs_lag_hours"].round(2)
pair_window_summary_df["median_signed_lag_hours_cluster2_minus_cluster1"] = pair_window_summary_df["median_signed_lag_hours_cluster2_minus_cluster1"].round(2)
if not quartile_pairing_summary_df.empty:
    quartile_pairing_summary_df["percent_cluster1_events_with_partner"] = quartile_pairing_summary_df["percent_cluster1_events_with_partner"].round(2)
    quartile_pairing_summary_df["median_abs_lag_hours"] = quartile_pairing_summary_df["median_abs_lag_hours"].round(2)
representative_pairs_df["lag_hours_cluster2_minus_cluster1"] = representative_pairs_df["lag_hours_cluster2_minus_cluster1"].round(2)
representative_pairs_df["abs_lag_hours"] = representative_pairs_df["abs_lag_hours"].round(2)

print("Cross-cluster pairing summary by window")
display(pair_window_summary_df)
print(f"\nRepresentative close pairs (up to {MAX_REPRESENTATIVE_PAIRS})")
display(representative_pairs_df.head(MAX_REPRESENTATIVE_PAIRS))
if quartile_overlay_available and not quartile_pairing_summary_df.empty:
    print("\nCluster 2 pairing frequency by Cluster 1 quartile subset")
    display(quartile_pairing_summary_df)


In [ ]:
required_globals = [
    "monthly_cluster_counts_df",
    "monthly_cluster1_quartile_counts_df",
    "pair_window_summary_df",
    "nearest_cluster1_df",
    "quartile_pairing_summary_df",
    "quartile_overlay_available",
    "plot_monthly_cluster_counts",
    "plot_quartile_pairing_rates",
    "plot_nearest_lag_histogram",
    "maybe_copy_to_drive",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 19 pairing-analysis cell before the plotting cell. "
        f"Missing globals: {missing_globals}"
    )

plot_inventory_rows = []

fig_monthly = plot_monthly_cluster_counts(monthly_cluster_counts_df)
monthly_plot_path = PLOT_DIR / "cleaned_k2_monthly_cluster_counts.png"
fig_monthly.savefig(monthly_plot_path, dpi=160, bbox_inches="tight")
plt.show()
maybe_copy_to_drive(monthly_plot_path)
plot_inventory_rows.append({"plot_kind": "bar", "plot_name": "monthly_cluster_counts", "local_path": str(monthly_plot_path), "drive_path": str(Path(DRIVE_OUTPUT_DIR) / monthly_plot_path.name)})

if quartile_overlay_available and not quartile_pairing_summary_df.empty:
    fig_quartile = plot_quartile_pairing_rates(quartile_pairing_summary_df)
    quartile_plot_path = PLOT_DIR / "cleaned_k2_cluster1_quartile_pairing_rates.png"
    fig_quartile.savefig(quartile_plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    maybe_copy_to_drive(quartile_plot_path)
    plot_inventory_rows.append({"plot_kind": "bar", "plot_name": "cluster1_quartile_pairing_rates", "local_path": str(quartile_plot_path), "drive_path": str(Path(DRIVE_OUTPUT_DIR) / quartile_plot_path.name)})

fig_lag = plot_nearest_lag_histogram(nearest_cluster1_df, max_window_days=MAX_PAIR_WINDOW_DAYS)
lag_plot_path = PLOT_DIR / f"cleaned_k2_nearest_lag_histogram_within_{MAX_PAIR_WINDOW_DAYS}d.png"
fig_lag.savefig(lag_plot_path, dpi=160, bbox_inches="tight")
plt.show()
maybe_copy_to_drive(lag_plot_path)
plot_inventory_rows.append({"plot_kind": "histogram", "plot_name": "nearest_cluster1_lag_histogram", "local_path": str(lag_plot_path), "drive_path": str(Path(DRIVE_OUTPUT_DIR) / lag_plot_path.name)})

plot_inventory_df = pd.DataFrame(plot_inventory_rows)
plot_inventory_df.to_csv(PLOT_INVENTORY_PATH, index=False)
maybe_copy_to_drive(PLOT_INVENTORY_PATH)
print("Saved Notebook 19 date-pairing plots")
display(plot_inventory_df)


In [ ]:
required_globals = [
    "context_summary_df",
    "quartile_count_df",
    "pair_window_summary_df",
    "representative_pairs_df",
    "quartile_overlay_available",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 19 plotting cell before the summary cell. "
        f"Missing globals: {missing_globals}"
    )

summary_pair_window_df = pair_window_summary_df.copy()
summary_pair_window_df["median_abs_lag_hours"] = summary_pair_window_df["median_abs_lag_hours"].round(2)
summary_pair_window_df["median_signed_lag_hours_cluster2_minus_cluster1"] = summary_pair_window_df["median_signed_lag_hours_cluster2_minus_cluster1"].round(2)

summary_representative_pairs_df = representative_pairs_df.head(MAX_REPRESENTATIVE_PAIRS).copy()
summary_representative_pairs_df = summary_representative_pairs_df[
    [
        "cluster1_event_index",
        "cluster1_event_peak_jst",
        "cluster1_quartile_subset",
        "cluster2_event_index",
        "cluster2_event_peak_jst",
        "lag_hours_cluster2_minus_cluster1",
        "abs_lag_hours",
        "within_max_window",
        "selection_source",
    ]
]

print("Notebook 19 date-pairing summary")
display(context_summary_df)
print("\nCluster 1 quartile-subset counts")
display(quartile_count_df)
print("\nCross-cluster pairing summary by window")
display(summary_pair_window_df)
if quartile_overlay_available and not quartile_pairing_summary_df.empty:
    print("\nCluster 2 pairing frequency by Cluster 1 quartile subset")
    display(quartile_pairing_summary_df)
print(f"\nRepresentative close pairs (up to {MAX_REPRESENTATIVE_PAIRS})")
display(summary_representative_pairs_df)
print("\nNotebook 19 now does the following:")
print("- quantifies cleaned Cluster 1 and Cluster 2 event pairing within ±1, ±2, and ±3 days")
print("- tracks whether Cluster 2 tends to occur before or after Cluster 1 in those close pairs")
print("- overlays the Notebook 18 lower / middle / upper Cluster 1 quartile tags when available")
print("- saves pair tables, nearest-pair summaries, monthly counts, and plot files for the next interpretation step")
